# Synthetic TV–TV reconstruction

A versioned 1024 master supplies the clean input. `CONFIG` selects the observation and reconstruction grid (256×256 by default). Clean block averaging precedes noise. The image grid and alpha partition are independent.

Each solve stage has an explicit switch, initially `False`. Preparation generates observations and previews without reconstruction. The configured depth-one partition is constant or contains four spatial children.


In [ ]:
from pathlib import Path
import json, subprocess, sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "configs/synthetic_256.json").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the updated project directory.")
sys.path.insert(0, str(ROOT / "src"))
from tvtv_numerics.original_1024 import Original1024
CONFIG = ROOT / "configs/synthetic_256.json"
settings = json.loads(CONFIG.read_text())
MASTER = ROOT / settings["data"]["master"]
print("Project:", ROOT)
print("Master:", MASTER)

def run_stage(action, minutes=30):
    command = [sys.executable, "-u", str(ROOT / "scripts/run_original_1024.py"),
               action, "--config", str(CONFIG), "--budget-minutes", str(minutes), "--seconds", str(minutes*60)]
    process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        # The launcher owns solver process groups; give it a chance to stop them.
        if sys.platform == "win32":
            subprocess.run(["taskkill", "/F", "/T", "/PID", str(process.pid)], capture_output=True)
        else:
            import signal
            process.send_signal(signal.SIGINT)
        process.wait()
        raise
    if code:
        print("Stage stopped. Inspect the output; completed checkpoints can be resumed.")

def current_report():
    experiment = Original1024(ROOT, CONFIG)
    return experiment, experiment.report()


## 1. Prepare observations and inspect the sequence

This cell prepares four noise seeds and estimates motion from observations. It starts no reconstruction. The master and archived proof-of-concept inputs remain separate.


In [ ]:
run_stage("prepare")
experiment, report = current_report()
with np.load(MASTER, allow_pickle=False) as z:
    master = z["reference"].copy()
from tvtv_numerics.data import load_sequence
data, _, _ = load_sequence(experiment.input_path(settings["data"]["design_seed"]))
fig, axes = plt.subplots(1, 5, figsize=(15, 3), constrained_layout=True)
for i, ax in enumerate(axes):
    ax.imshow(master[i], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Master frame {i}")
    ax.axis("off")
plt.show()
fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
for ax, image, title in zip(axes, [master[0], data.reference[0], data.noisy[0]],
                            ["Clean 1024 master", "Averaged clean working grid", "Noise added on working grid"]):
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
plt.show()
print("Working shape:", data.reference.shape, "; noise std:", settings["data"]["noise_std"])


## 2. Learn a constant alpha

Start with three values, expand the range if its endpoint wins, and test geometric midpoints around the interior winner. This is a bounded search, not a certified continuous optimum. Every alpha starts from the same observation-based initialization.

Set `RUN_BASELINE = True` and run this cell. A ten-minute budget applies to this invocation. If it stops because of time, run the same cell again; do not discard the checkpoint. If it reports a maximum iteration limit, numerical diagnostics must be reviewed.


In [ ]:
RUN_BASELINE = False
if RUN_BASELINE:
    run_stage("baseline", minutes=10)
experiment, report = current_report()
root = report["learning"]["root"]
for row in root["rows"]:
    print("alpha:", row["alpha"], "status:", row["status"], "loss:", row["upper_loss"])
if root["complete"]:
    print("Selected alpha:", root["selected"]["alpha"], "; boundary limited:", root["boundary_limited"])


## 3. Learn the spatial partition

Set `RUN_REFINE = True` only after the constant search has completed. A parent splits into four children only if their summed learned crop loss is below `rho × parent loss`. All crops keep the original spacing; no temporal splits or halos are used. Search includes the parent's alpha. The current configurations allow at most one split into four spatial children.

A rejected split is recorded. Do not force acceptance to obtain a desired picture.


In [ ]:
RUN_REFINE = False
if RUN_REFINE:
    run_stage("refine", minutes=30)
experiment, report = current_report()
learning = report["learning"]
print("Complete:", learning["complete"], "; blocked:", learning.get("blocked"))
for decision in learning["decisions"]:
    print("Cell", decision["cell"], ": accepted", decision["accepted"],
          "; parent loss", decision["parent_loss"], "; children loss", decision["children_loss"])
if learning["complete"]:
    alpha_map = np.zeros(data.reference.shape[1:])
    for leaf in learning["leaves"]:
        level, t0, t1, y0, y1, x0, x1 = leaf["cell"]
        alpha_map[y0:y1, x0:x1] = leaf["alpha"]
    fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
    im = ax.imshow(alpha_map, cmap="viridis")
    ax.set_title(f"Accepted partition: {len(learning['leaves'])} leaves")
    fig.colorbar(im, ax=ax, label="alpha")
    plt.show()


## 4. Global comparison on the design seed

Set `RUN_COMPARE = True` after refinement completes. Both the learned constant and the accepted spatial map receive a full-domain reconstruction at tighter tolerances, with the same noisy input and initialization. If the first split was rejected, its proposed map is also checked as a separately labelled global diagnostic; it is not the accepted partition.


In [ ]:
RUN_COMPARE = False
if RUN_COMPARE:
    run_stage("compare", minutes=15)
experiment, report = current_report()
pair = report["comparisons"][0] if report["comparisons"] else None
if pair and pair["complete"]:
    print("Design loss gain:", 100*pair["relative_gain"], "%")
    fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
    axes[0].imshow(data.reference[0], cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("Clean working reference")
    for ax, kind in zip(axes[1:], ("constant", "spatial")):
        row = pair[kind]
        job = {k: row[k] for k in ("stage", "seed", "cell", "alpha", "mode", "leaves")}
        _, arrays = experiment.inspect(job, with_arrays=True)
        ax.imshow(arrays[0][0], cmap="gray", vmin=0, vmax=1)
        ax.set_title("Constant" if kind == "constant" else f"Accepted map ({len(report['learning']['leaves'])} leaves)")
    for ax in axes:
        ax.axis("off")
    plt.show()
else:
    print("Global comparison is not complete yet.")
diagnostic = report.get("rejected_split_global_diagnostic")
if diagnostic and diagnostic["eligible"] and pair and pair["constant"]["eligible"]:
    gain = 1-diagnostic["upper_loss"]/pair["constant"]["upper_loss"]
    print("Rejected first split: diagnostic global loss gain", 100*gain, "%")


## 5. Validate the frozen model on three fresh noise seeds

Set `RUN_VALIDATE = True` only after reviewing the design comparison. The constant alpha and partition are frozen from the design seed. Fresh clean references are used to score results, not to tune the parameters again. A finite design search can still leave a stronger constant baseline undiscovered.


In [ ]:
RUN_VALIDATE = False
if RUN_VALIDATE:
    run_stage("validate", minutes=30)
experiment, report = current_report()
for pair in report["comparisons"]:
    if pair["complete"]:
        print("Seed", pair["seed"], ": loss gain", 100*pair["relative_gain"],
              "%; RMSE gain", 100*pair["RMSE_relative_gain"], "%")
    else:
        print("Seed", pair["seed"], ": pending")


## 6. Export the experiment

Set the switch to `True` to export inputs, protocol, candidates, diagnostics and saved reconstructions. Export starts no solver. The archive path is printed. Results remain under `results/synthetic_original_1024/<protocol-id>/`.


In [ ]:
EXPORT = False
if EXPORT:
    run_stage("export")
print("Results:", experiment.out)
